## [1] Setup & Paths

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import sys

# Rutas corregidas según tus datos reales
PATH_TEAM_GAMELOGS = Path("/Users/pablo/Documents/BigData/BasketballAnalysis/00_data/00d_featurized/2024-25/teamgamelogs_featurized.parquet")
PATH_BOXSCORES = Path("/Users/pablo/Documents/BigData/BasketballAnalysis/00_data/00c_final/2024-25/boxscores.parquet")  # Tu archivo real
PATH_ON = None  # Omitir si no existe
PATH_OFF = None  # Omitir si no existe  
PATH_LINEUPS = None  # Omitir si no existe

print("Boxscores columns:", pd.read_parquet(PATH_BOXSCORES, nrows=0).columns.tolist())

## [2] Cargar DataFrames

In [ ]:
# Cargar datos principales
df_team = pd.read_parquet(PATH_TEAM_GAMELOGS)
df_box = pd.read_parquet(PATH_BOXSCORES)

print(f"Team games: {df_team.shape}")
print(f"Boxscores: {df_box.shape}")
print("Team columns:", df_team.columns.tolist()[:10])

## [3] Importar funciones

In [ ]:
# Añadir path para importar
sys.path.append('/Users/pablo/Documents/BigData/BasketballAnalysis/02_processing_data/02a_WL_prediction')

import importlib.util
module_path = Path('/Users/pablo/Documents/BigData/BasketballAnalysis/02_processing_data/02a_WL_prediction/02_FeatureFunctions.py')
spec = importlib.util.spec_from_file_location('feature_functions', module_path)
feature_functions = importlib.util.module_from_spec(spec)
spec.loader.exec_module(feature_functions)

from feature_functions import add_lineup_features_in_memory, DEFAULT_LINEUP_CONFIG

## [4] Construcción in-memory de LINEUP_*

In [ ]:
# Configuración inicial - sin on/off ni lineups
config = DEFAULT_LINEUP_CONFIG.copy()
config.update({
    'USE_ON_OFF': False,  # No tenemos estos datos
    'USE_LINEUPS': False  # No tenemos estos datos
})

df_aug = add_lineup_features_in_memory(
    df_teamgames=df_team,
    df_player_box=df_box,
    df_on=PATH_ON,
    df_off=PATH_OFF, 
    df_lineups=PATH_LINEUPS,
    config=config
)

print("Columnas añadidas:", [col for col in df_aug.columns if 'LINEUP' in col])
print(df_aug.filter(like='LINEUP_').describe())

## [5] Sanity checks + correlaciones

In [ ]:
# Verificar nulos
lineup_cols = [col for col in df_aug.columns if 'LINEUP' in col]
print("NaN rates:")
print(df_aug[lineup_cols].isna().mean())

# Correlación con victorias (si existe columna W)
if 'W' in df_aug.columns:
    correlations = df_aug[lineup_cols].corrwith(df_aug['W']).sort_values(ascending=False)
    print("Correlaciones con W:")
    print(correlations)

## [6] Conclusiones

- Evaluar la calidad de las features `LINEUP_*`.
- Verificar que no hay leakage temporal.
- Documentar próximos pasos.